This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each ..05 of similarity scores
- then, get cohen's kappas on that for comparison

In [1]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix


Welcome to NLP eval for DE, version 1.0.0


In [109]:
testable_data = data.get_testable_data("Example\\inputs\\case study 2 input-open codes\\unclean version\\hackathon numerical GTs.csv")
codes = data.get_codes("Example\\inputs\\case study 2 input-open codes\\hackathon open codes.csv")
all_scores = scores.get_BART_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
#make this go up to 88
all_scores_expanded[[str(i) for i in range(1, 89)]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Open code(s)" column of testable_data to all_scores_expanded
all_scores_expanded["Open code(s)"] = testable_data["Open code(s)"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

Device set to use cuda:0


,Input phrase,1,2,3,4,5,6,7,8,9,...,80,81,82,83,84,85,86,87,88,Open code(s)
0,Do you want to do like a mic check real quick or?,0.523516,0.068094,0.177669,0.854539,0.568454,0.900970,0.286383,0.984132,0.410779,...,0.109396,0.425900,0.465372,0.921246,0.675838,0.101516,0.313971,0.092440,0.230452,0
1,Is that correct?,0.498506,0.074379,0.458651,0.894089,0.433128,0.794634,0.449798,0.696069,0.329936,...,0.314380,0.345200,0.515204,0.625458,0.412278,0.526193,0.711321,0.103677,0.511112,0
2,"OK, testing 123123. Testing. Testing.",0.743834,0.134737,0.607001,0.967034,0.494921,0.924647,0.518822,0.924769,0.943019,...,0.040055,0.539795,0.816445,0.702619,0.381516,0.227331,0.854187,0.000844,0.052369,0
3,Yeah.,0.547771,0.309241,0.616967,0.832713,0.741985,0.876919,0.677319,0.894702,0.732099,...,0.443559,0.559290,0.805495,0.811386,0.705275,0.733567,0.818673,0.312974,0.656928,0
4,"Yeah, 123123.",0.630403,0.076005,0.615024,0.786398,0.704545,0.822700,0.531645,0.855875,0.604466,...,0.210303,0.403498,0.869682,0.728040,0.660737,0.511351,0.824253,0.011650,0.215889,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
901,"Yeah, I'll do.",0.641846,0.286043,0.741552,0.911969,0.856071,0.973474,0.865365,0.980903,0.820879,...,0.621983,0.765223,0.979634,0.792431,0.870985,0.618159,0.837477,0.209004,0.337457,0
902,"Well, we may have to ask [them] about about that?",0.523257,0.134560,0.492838,0.964394,0.411591,0.874278,0.457475,0.869085,0.593936,...,0.297415,0.481182,0.457205,0.728330,0.425496,0.257372,0.712454,0.134227,0.622329,0
903,Yeah. OK.,0.565039,0.302818,0.564094,0.829154,0.786483,0.853352,0.719215,0.876998,0.692798,...,0.402153,0.526555,0.791656,0.768956,0.615600,0.765901,0.798632,0.318649,0.567705,0
904,So this is Speaker 4,0.164569,0.000734,0.059474,0.149393,0.025557,0.356032,0.191517,0.045448,0.014858,...,0.015852,0.006696,0.845939,0.160521,0.104975,0.010518,0.121750,0.000268,0.001105,0


In [110]:
def eval_filtered(all_scores_expanded, min, max):
    # now filter based on threshold 
    # drop any rows where the highest score from the 10 scores is not between min and max
    all_scores_expanded_filtered = all_scores_expanded[
        (all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) >= min) 
        & (all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) <= max)]

    # now, get cohens kappa score for the filtered data
    ground_truths = all_scores_expanded_filtered["Open code(s)"].tolist()
    predictions = all_scores_expanded_filtered[[str(i) for i in range(1, 89)]].idxmax(axis=1).tolist()
    # convert predictions from strings to ints
    predictions = [int(x) for x in predictions]
    #make these labels also go to 88 instead of 11
    f1s = f1_score(ground_truths, predictions, labels=list(range(1, 89)), average=None, zero_division=0.0) 
    #mtx = confusion_matrix(ground_truths, predictions, labels=list(range(1, 89)))
    kappa = cohen_kappa_score(ground_truths, predictions, weights=None, sample_weight=None)
    # get average f1 (will see later if this is ok)
    f1 = sum(f1s) / len(f1s)
    # count rows below the minimum threshold
    rows_below_min = len(all_scores_expanded[all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) < min])
    return [f1, kappa, rows_below_min]

In [111]:
rows = []
total_rows = len(all_scores_expanded)
for i, j in [(0.95, 1.0), (0.9, 0.95), (0.85, 0.9), (0.8, 0.85), (0.75, 0.8), (0.7, 0.75), (0.65, 0.7), (0.6, 0.65), (0.55, 0.6), (0.5, 0.55), (0.45, 0.5), (0.4, 0.45), (0.35, 0.4), (0.3, 0.35), (0.25, 0.3), (0.2, 0.25), (0.15, 0.2), (0.1, 0.15), (0.05, 0.1), (0.0, 0.05)]:
    results = eval_filtered(all_scores_expanded, i, j)
    rows_below_min = results[2]
    percentage_below_min = (rows_below_min / total_rows) * 100
    rows.append({"min": i, "max": j, "kappa": results[1], "f1": results[0], "percentage_below_min": percentage_below_min, "rows_below_min": rows_below_min})
thresholded_kappas = pd.DataFrame(rows)
thresholded_kappas.to_csv("thresholding_results\\thrKappas_BART_open_unclean_hackathon.csv", index=False)
thresholded_kappas

C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeW

,min,max,kappa,f1,percentage_below_min,rows_below_min
0,0.95,1.00,0.039640,0.067355,29.801325,270
1,0.90,0.95,0.001648,0.001337,10.154525,92
2,0.85,0.90,-0.001669,0.000000,4.746137,43
3,0.80,0.85,0.000000,0.000000,2.869757,26
4,0.75,0.80,-0.012500,0.000000,1.876380,17
5,0.70,0.75,-0.008333,0.000000,0.662252,6
6,0.65,0.70,0.000000,0.000000,0.331126,3
7,0.60,0.65,0.000000,0.000000,0.220751,2
8,0.55,0.60,0.000000,0.000000,0.110375,1
9,0.50,0.55,NaN,0.000000,0.110375,1
